# HomeBot-Sim: fine-tune GR00T N1.7 (frozen VLM, fp16, 2x T4)
Day-1 gate: run the 20-step VRAM probe first. If it OOMs, ACT is the primary policy.

In [ ]:
import os, subprocess
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
DATA = "/kaggle/input/homebot-can"          # Kaggle Dataset made from data/homebot_can
!nvidia-smi

In [ ]:
!git clone --depth 1 https://github.com/NVIDIA/Isaac-GR00T /kaggle/working/Isaac-GR00T
%cd /kaggle/working/Isaac-GR00T
!pip install -q -e . && pip install -q flash-attn --no-build-isolation || true
!git clone --depth 1 https://github.com/<your-user>/homebot-sim /kaggle/working/homebot-sim

## 20-step VRAM probe

In [ ]:
import sys; sys.path.insert(0, "/kaggle/working/homebot-sim")
from configs.gr00t_g1_sonic_inspire import TRAIN
cmd = f'''python scripts/gr00t_finetune.py --dataset-path {DATA} --num-gpus 1
  --output-dir /kaggle/working/probe --max-steps 20 --batch-size {TRAIN["batch_size"]}
  --no-tune-llm --no-tune-visual --tune-projector --tune-diffusion-model
  --data-config /kaggle/working/homebot-sim/configs/gr00t_g1_sonic_inspire.py:make_config
  --embodiment-tag new_embodiment --report-to none'''
!{cmd} 2>&1 | tail -20
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv

## Full fine-tune (checkpoint every 500 steps -> HF Hub, private)

In [ ]:
cmd = cmd.replace("--max-steps 20", f"--max-steps {TRAIN['max_steps']} --save-steps 500").replace("/probe", "/n17_homebot")
!{cmd} 2>&1 | tail -50
!huggingface-cli upload --private <your-user>/homebot-n17 /kaggle/working/n17_homebot .